# iPad RER Robustness Checks — Full Replication Notebook

This notebook runs the robustness checks used in the revised iPad real exchange rate paper.

It produces:

1. **Table 1** — coverage of restrictive windows  
2. **Table 2** — strict no-price-reset product-level and pooled estimates  
3. **Table 3** — pooled panel identification checks  
4. **Table 4** — pooled panel sensitivity checks  

The notebook is designed to avoid the mask-alignment problems that caused blank pooled-panel rows in earlier versions.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm


In [2]:
# ============================================================
# USER SETTINGS
# ============================================================

# If hfdata3.csv is in the same folder as this notebook, leave as is.
# Otherwise, change this to the full path, e.g. Path(r"C:/Users/.../hfdata3.csv")
CSV_PATH = Path("hfdata3.csv")

# Fallback for ChatGPT / sandbox runs
if not CSV_PATH.exists():
    fallback = Path("/mnt/data/hfdata3.csv")
    if fallback.exists():
        CSV_PATH = fallback

OUTPUT_XLSX = Path("ipad_robustness_tables_full_replication.xlsx")
OUTPUT_LATEX = Path("ipad_robustness_tables_full_replication.tex")

BASE_COUNTRY = "US"
D_WINDOWS = [12, 24, 36, 48]

# COVID split used in the robustness table.
# Change this if the paper uses a different cutoff.
COVID_SPLIT_DATE = pd.Timestamp("2020-01-01")

HIGH_RESET_COUNTRIES = ["BR", "MX", "RU", "TR"]

PRODUCTS = {
    "iPad Pro Large": {
        "model_col": "ipadpro12.9",
        "price_col": "ipadpro12.9p",
    },
    "iPad Pro Small": {
        "model_col": "ipadprosmall",
        "price_col": "ipadprosmallp",
    },
    "iPad": {
        "model_col": "ipad",
        "price_col": "ipadp",
    },
    "iPad Mini": {
        "model_col": "ipadm",
        "price_col": "ipadmp",
    },
}

print("CSV path:", CSV_PATH)


CSV path: hfdata3.csv


## Helper functions

In [3]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def stars(p):
    """Return significance stars from a p-value."""
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def fmt_coef(beta, p):
    """Format coefficient with significance stars."""
    if pd.isna(beta):
        return ""
    return f"{beta:.3f}{stars(p)}"


def safe_numeric(s):
    """
    Convert messy numeric columns to float.
    Invalid entries become NaN.
    Handles non-breaking spaces and comma separators.
    """
    return pd.to_numeric(
        s.astype(str)
         .str.replace("\xa0", "", regex=False)
         .str.replace(",", "", regex=False)
         .str.strip(),
        errors="coerce"
    )


def read_csv_robust(path):
    """Read CSV with a fallback encoding."""
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin1")


## Build the product-country-week panel

In [4]:
def build_long_panel(raw):
    """
    Convert the wide iPad database into a long product-country-week panel.

    Constructs:
        q = log(local price) - log(US price) - log(exchange rate)
        q_tilde = q - country-product mean(q)

    Notes:
    - The US is used as the base country.
    - Non-US observations are retained.
    """

    raw = raw.copy()
    raw.columns = [c.strip().replace("\xa0", " ") for c in raw.columns]

    raw["date"] = pd.to_datetime(raw["Date"], dayfirst=True, errors="coerce")
    raw["Country"] = raw["Country"].astype(str).str.strip()
    raw["Currency"] = raw["Currency"].astype(str).str.strip()
    raw["ER"] = safe_numeric(raw["ER"])

    frames = []

    for product, spec in PRODUCTS.items():
        model_col = spec["model_col"]
        price_col = spec["price_col"]

        tmp = raw[
            ["date", "Country", "Currency", "Week", "ER", model_col, price_col]
        ].copy()

        tmp = tmp.rename(
            columns={
                "Country": "country",
                model_col: "model",
                price_col: "price",
            }
        )

        tmp["product"] = product
        tmp["model"] = (
            tmp["model"]
            .astype(str)
            .str.replace("\xa0", " ", regex=False)
            .str.strip()
            .replace({"nan": np.nan, "": np.nan})
        )
        tmp["price"] = safe_numeric(tmp["price"])

        tmp = tmp.dropna(subset=["date", "country", "price", "ER"])
        tmp = tmp[(tmp["price"] > 0) & (tmp["ER"] > 0)]

        # US base price/model by date/product
        us = tmp[tmp["country"] == BASE_COUNTRY][
            ["date", "product", "price", "model"]
        ].rename(
            columns={
                "price": "us_price",
                "model": "us_model",
            }
        )

        tmp = tmp.merge(us, on=["date", "product"], how="left")

        # Exclude US from country sample; require US benchmark price
        tmp = tmp[tmp["country"] != BASE_COUNTRY].copy()
        tmp = tmp.dropna(subset=["us_price"])

        tmp["q"] = np.log(tmp["price"]) - np.log(tmp["us_price"]) - np.log(tmp["ER"])
        tmp["country_product"] = tmp["country"] + "_" + tmp["product"]

        tmp["qbar"] = tmp.groupby(["country", "product"])["q"].transform("mean")
        tmp["q_tilde"] = tmp["q"] - tmp["qbar"]

        frames.append(tmp)

    panel = pd.concat(frames, ignore_index=True)
    panel = panel.sort_values(["product", "country", "date"]).reset_index(drop=True)

    return panel


## Add horizons and restriction flags

In [5]:
def add_no_change_flags(g, col, d, new_col):
    """
    Flag observations where `col` has not changed at any weekly transition
    from t-d to t.

    This is stricter than checking only value_t == value_t-d.
    """
    g = g.sort_values("date").copy()

    change = g[col].ne(g[col].shift(1)).fillna(False).astype(int)
    if len(change) > 0:
        change.iloc[0] = 0

    g[new_col] = change.rolling(d, min_periods=d).sum().eq(0)
    return g


def add_horizon_variables(panel, d, q_col="q_tilde", suffix=""):
    """
    Add:
    - lagged q
    - d-week change in q
    - sign-adjusted dependent variable
    - absolute lagged deviation
    - valid d-week window flag
    - local / US price fixed flags
    - local / US model fixed flags
    """
    df = panel.copy().sort_values(["country", "product", "date"]).reset_index(drop=True)
    group_cols = ["country", "product"]

    lag_name = f"q_lag_{d}{suffix}"
    dq_name = f"dq_{d}{suffix}"
    y_name = f"y_sign_{d}{suffix}"
    abs_name = f"abs_lag_{d}{suffix}"

    df[lag_name] = df.groupby(group_cols)[q_col].shift(d)
    df[dq_name] = df[q_col] - df[lag_name]
    df[abs_name] = df[lag_name].abs()
    df[y_name] = np.sign(df[lag_name]) * df[dq_name]

    df[f"date_lag_{d}"] = df.groupby(group_cols)["date"].shift(d)
    df[f"valid_window_{d}"] = (
        (df["date"] - df[f"date_lag_{d}"]).dt.days == 7 * d
    )

    # Strict no-change flags over the full d-week window
    df = (
        df.groupby(group_cols, group_keys=False)
        .apply(lambda g: add_no_change_flags(g, "price", d, f"local_price_fixed_{d}"))
    )

    df = (
        df.groupby(group_cols, group_keys=False)
        .apply(lambda g: add_no_change_flags(g, "us_price", d, f"us_price_fixed_{d}"))
    )

    df = (
        df.groupby(group_cols, group_keys=False)
        .apply(lambda g: add_no_change_flags(g, "model", d, f"local_model_fixed_{d}"))
    )

    df = (
        df.groupby(group_cols, group_keys=False)
        .apply(lambda g: add_no_change_flags(g, "us_model", d, f"us_model_fixed_{d}"))
    )

    df[f"both_prices_fixed_{d}"] = (
        df[f"local_price_fixed_{d}"] & df[f"us_price_fixed_{d}"]
    )

    df[f"same_model_window_{d}"] = (
        df[f"local_model_fixed_{d}"] & df[f"us_model_fixed_{d}"]
    )

    df = df.sort_values(["product", "country", "date"]).reset_index(drop=True)
    return df


## Threshold regression engine

This is the corrected version. It aligns masks by index and works for product-level and pooled regressions.


In [6]:
def threshold_regression(
    df,
    d,
    sample_mask=None,
    product=None,
    pooled=False,
    q_suffix="",
    min_obs=100,
    min_regime_share=0.05,
    grid_size=120,
    debug=False,
):
    """
    Estimate the sign-adjusted threshold robustness regression:

        sign(q_lag) * Δq
        = rho0 * |q_lag| * 1{|q_lag| <= c}
        + rho1 * |q_lag| * 1{|q_lag| > c}
        + product fixed effects if pooled
        + error

    More negative coefficients imply faster corrective dynamics.

    The threshold c is chosen by grid search over interior values of |q_lag|.
    """

    y_col = f"y_sign_{d}{q_suffix}"
    x_col = f"abs_lag_{d}{q_suffix}"

    dat = df if product is None else df[df["product"] == product]

    base_mask = (
        dat[f"valid_window_{d}"].fillna(False)
        & dat[y_col].notna()
        & dat[x_col].notna()
        & np.isfinite(dat[y_col])
        & np.isfinite(dat[x_col])
    )

    if sample_mask is not None:
        if isinstance(sample_mask, pd.Series):
            smask = sample_mask.reindex(dat.index).fillna(False).astype(bool)
        else:
            smask = pd.Series(sample_mask, index=dat.index).astype(bool)
        base_mask = base_mask & smask

    dat = dat.loc[base_mask].copy()

    if debug:
        print(f"d={d}, product={product}, pooled={pooled}, observations={len(dat)}")

    if len(dat) < min_obs:
        return None

    x = dat[x_col].astype(float)

    # Interior grid: avoids thresholds with vanishingly small regimes.
    lo, hi = np.nanpercentile(x, [10, 90])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return None

    grid = np.linspace(lo, hi, grid_size)

    best = None

    for c in grid:
        inner_mask = (x <= c).astype(float)
        outer_mask = (x > c).astype(float)

        if inner_mask.mean() < min_regime_share or outer_mask.mean() < min_regime_share:
            continue

        X = pd.DataFrame(
            {
                "inner": x * inner_mask,
                "outer": x * outer_mask,
            },
            index=dat.index,
        ).astype(float)

        if pooled:
            dummies = pd.get_dummies(dat["product"], prefix="prod", drop_first=True).astype(float)
            X = pd.concat([X, dummies], axis=1).astype(float)

        try:
            fit = sm.OLS(dat[y_col].astype(float), X, missing="drop").fit()
            ssr = float(np.sum(fit.resid ** 2))
        except Exception:
            continue

        if best is None or ssr < best["ssr"]:
            best = {
                "c": float(c),
                "ssr": ssr,
                "X": X,
                "dat": dat,
            }

    if best is None:
        return None

    X = best["X"].astype(float)
    dat = best["dat"]

    groups = dat["country_product"] if pooled else dat["country"]

    try:
        fit = sm.OLS(dat[y_col].astype(float), X, missing="drop").fit(
            cov_type="cluster",
            cov_kwds={"groups": groups, "use_correction": True},
        )
    except Exception:
        fit = sm.OLS(dat[y_col].astype(float), X, missing="drop").fit()

    params = fit.params
    pvals = fit.pvalues

    p_equal = np.nan
    try:
        names = list(params.index)
        R = np.zeros((1, len(names)))
        R[0, names.index("outer")] = 1
        R[0, names.index("inner")] = -1
        p_equal = float(fit.t_test(R).pvalue)
    except Exception:
        pass

    return {
        "d": d,
        "nobs": int(fit.nobs),
        "threshold": best["c"],
        "rho_inner": params.get("inner", np.nan),
        "rho_outer": params.get("outer", np.nan),
        "p_inner": pvals.get("inner", np.nan),
        "p_outer": pvals.get("outer", np.nan),
        "p_equal": p_equal,
        "r2": fit.rsquared,
    }


def retained_share(df, d, mask, product=None, q_suffix=""):
    """Share of valid d-week windows retained under a restriction."""
    dat = df if product is None else df[df["product"] == product]

    y_col = f"y_sign_{d}{q_suffix}"
    x_col = f"abs_lag_{d}{q_suffix}"

    base = (
        dat[f"valid_window_{d}"].fillna(False)
        & dat[y_col].notna()
        & dat[x_col].notna()
    )

    denom = int(base.sum())
    if denom == 0:
        return np.nan

    if isinstance(mask, pd.Series):
        smask = mask.reindex(dat.index).fillna(False).astype(bool)
    else:
        smask = pd.Series(mask, index=dat.index).astype(bool)

    num = int((base & smask).sum())
    return 100 * num / denom


def result_row(label, res, retained=None):
    """Convert regression result into formatted table row."""
    if res is None:
        return {
            "Sample": label,
            "d": "",
            "Inner rho0": "",
            "Outer rho1": "",
            "c": "",
            "Retained windows": "" if retained is None else f"{retained:.1f}%",
            "p(rho1=rho0)": "",
        }

    return {
        "Sample": label,
        "d": res["d"],
        "Inner rho0": fmt_coef(res["rho_inner"], res["p_inner"]),
        "Outer rho1": fmt_coef(res["rho_outer"], res["p_outer"]),
        "c": f"{res['threshold']:.3f}",
        "Retained windows": "" if retained is None else f"{retained:.1f}%",
        "p(rho1=rho0)": "" if pd.isna(res["p_equal"]) else f"{res['p_equal']:.3f}",
    }


## Load data and construct all variables

In [7]:
raw = read_csv_robust(CSV_PATH)

print("Raw shape:", raw.shape)
print("Columns:", list(raw.columns))

panel = build_long_panel(raw)
print("Long panel shape:", panel.shape)

df = panel.copy()
for d in D_WINDOWS:
    df = add_horizon_variables(df, d)

print("Analysis panel shape:", df.shape)
df.head()


Raw shape: (10990, 18)
Columns: ['Date', 'Country', 'Currency', 'ipadpro12.9', 'ipadpro12.9p', 'ipadprosmall', 'ipadprosmallp', 'ipad', 'ipadp', 'ipadm', 'ipadmp', 'vatgst', 'tariff', 'Source', 'Start Date', 'End Date', 'ER', 'Week']
Long panel shape: (42296, 14)
Analysis panel shape: (42296, 62)


,date,country,Currency,Week,ER,model,price,product,us_price,us_model,...,abs_lag_48,y_sign_48,date_lag_48,valid_window_48,local_price_fixed_48,us_price_fixed_48,local_model_fixed_48,us_model_fixed_48,both_prices_fixed_48,same_model_window_48
0,2016-01-01,AE,AED,1,3.673,iPad Air,1499.0,iPad,399.0,iPad Air,...,NaN,NaN,NaT,False,False,False,False,False,False,False
1,2016-01-08,AE,AED,2,3.673,iPad Air,1499.0,iPad,399.0,iPad Air,...,NaN,NaN,NaT,False,False,False,False,False,False,False
2,2016-01-15,AE,AED,3,3.673,iPad Air,1499.0,iPad,399.0,iPad Air,...,NaN,NaN,NaT,False,False,False,False,False,False,False
3,2016-01-22,AE,AED,4,3.673,iPad Air,1499.0,iPad,399.0,iPad Air,...,NaN,NaN,NaT,False,False,False,False,False,False,False
4,2016-01-29,AE,AED,5,3.673,iPad Air,1499.0,iPad,399.0,iPad Air,...,NaN,NaN,NaT,False,False,False,False,False,False,False


## Winsorized panel for Table 4

In [8]:
def winsorize_q(panel):
    """Winsorize q_tilde at the 1st and 99th percentiles within each country-product series."""
    out = panel.copy()

    def clip_group(s):
        lo, hi = s.quantile([0.01, 0.99])
        return s.clip(lo, hi)

    out["q_tilde_winsor"] = (
        out.groupby(["country", "product"])["q_tilde"]
        .transform(clip_group)
    )

    return out


panel_w = winsorize_q(panel)

df_w = panel_w.copy()
for d in D_WINDOWS:
    df_w = add_horizon_variables(df_w, d, q_col="q_tilde_winsor", suffix="_w")

print("Winsorized analysis panel shape:", df_w.shape)
df_w.head()


Winsorized analysis panel shape: (42296, 63)


,date,country,Currency,Week,ER,model,price,product,us_price,us_model,...,abs_lag_48_w,y_sign_48_w,date_lag_48,valid_window_48,local_price_fixed_48,us_price_fixed_48,local_model_fixed_48,us_model_fixed_48,both_prices_fixed_48,same_model_window_48
0,2016-01-01,AE,AED,1,3.673,iPad Air,1499.0,iPad,399.0,iPad Air,...,NaN,NaN,NaT,False,False,False,False,False,False,False
1,2016-01-08,AE,AED,2,3.673,iPad Air,1499.0,iPad,399.0,iPad Air,...,NaN,NaN,NaT,False,False,False,False,False,False,False
2,2016-01-15,AE,AED,3,3.673,iPad Air,1499.0,iPad,399.0,iPad Air,...,NaN,NaN,NaT,False,False,False,False,False,False,False
3,2016-01-22,AE,AED,4,3.673,iPad Air,1499.0,iPad,399.0,iPad Air,...,NaN,NaN,NaT,False,False,False,False,False,False,False
4,2016-01-29,AE,AED,5,3.673,iPad Air,1499.0,iPad,399.0,iPad Air,...,NaN,NaN,NaT,False,False,False,False,False,False,False


# Table 1 — Coverage of restrictive windows

In [9]:
coverage_rows = []

for product in list(PRODUCTS.keys()) + ["All 4 varieties"]:
    row = {"Product": product}

    for restriction in ["both_prices_fixed", "same_model_window"]:
        for d in D_WINDOWS:
            prod_arg = None if product == "All 4 varieties" else product
            mask = df[f"{restriction}_{d}"]
            row[f"{restriction}_d{d}"] = retained_share(df, d, mask, product=prod_arg)

    coverage_rows.append(row)

table1 = pd.DataFrame(coverage_rows)

table1_pretty = table1.copy()
for col in table1_pretty.columns:
    if col != "Product":
        table1_pretty[col] = table1_pretty[col].map(lambda x: f"{x:.1f}%")

table1_pretty


,Product,both_prices_fixed_d12,both_prices_fixed_d24,both_prices_fixed_d36,both_prices_fixed_d48,same_model_window_d12,same_model_window_d24,same_model_window_d36,same_model_window_d48
0,iPad Pro Large,83.6%,67.3%,51.4%,41.5%,100.0%,100.0%,100.0%,100.0%
1,iPad Pro Small,83.2%,67.0%,51.9%,39.3%,91.7%,82.7%,72.9%,62.2%
2,iPad,79.0%,63.3%,49.3%,35.7%,92.1%,87.6%,82.7%,77.4%
3,iPad Mini,82.2%,67.6%,55.6%,43.8%,92.1%,83.4%,74.1%,63.9%
4,All 4 varieties,82.0%,66.3%,52.1%,40.1%,94.0%,88.5%,82.5%,76.0%


# Table 2 — Strict no-price-reset sample

In [10]:
table2_rows = []

# Product-specific rows
for product in PRODUCTS.keys():
    for d in D_WINDOWS:
        mask = df[f"both_prices_fixed_{d}"]

        res = threshold_regression(
            df,
            d,
            sample_mask=mask,
            product=product,
            pooled=False,
        )

        retained = retained_share(df, d, mask, product=product)
        table2_rows.append(result_row(product, res, retained))

# Pooled all-product rows
for d in D_WINDOWS:
    mask = df[f"both_prices_fixed_{d}"]

    res = threshold_regression(
        df,
        d,
        sample_mask=mask,
        product=None,
        pooled=True,
    )

    retained = retained_share(df, d, mask, product=None)
    table2_rows.append(result_row("All 4 varieties", res, retained))

table2 = pd.DataFrame(table2_rows)
table2


,Sample,d,Inner rho0,Outer rho1,c,Retained windows,p(rho1=rho0)
0,iPad Pro Large,12,-0.123***,-0.284***,0.045,83.6%,0.002
1,iPad Pro Large,24,-0.215***,-0.633***,0.045,67.3%,0.000
2,iPad Pro Large,36,-0.481***,-1.042***,0.046,51.4%,0.000
3,iPad Pro Large,48,-0.265*,-1.258***,0.031,41.5%,0.000
4,iPad Pro Small,12,-0.080*,-0.269***,0.047,83.2%,0.002
5,iPad Pro Small,24,-0.149**,-0.525***,0.048,67.0%,0.000
6,iPad Pro Small,36,-0.444***,-0.871***,0.054,51.9%,0.000
7,iPad Pro Small,48,-0.302***,-1.062***,0.035,39.3%,0.000
8,iPad,12,-0.261**,-0.011,0.029,79.0%,0.015
9,iPad,24,-0.315**,-0.026,0.026,63.3%,0.053


# Table 3 — Pooled panel identification checks

In [11]:
table3_rows = []

sample_defs = [
    ("Full sample", lambda base, d: pd.Series(True, index=base.index)),
    ("Local price fixed", lambda base, d: base[f"local_price_fixed_{d}"]),
    ("US price fixed", lambda base, d: base[f"us_price_fixed_{d}"]),
    ("Both prices fixed", lambda base, d: base[f"both_prices_fixed_{d}"]),
    ("Same model window", lambda base, d: base[f"same_model_window_{d}"]),
]

for sample_name, mask_fun in sample_defs:
    for d in D_WINDOWS:
        mask = mask_fun(df, d)

        res = threshold_regression(
            df,
            d,
            sample_mask=mask,
            product=None,
            pooled=True,
        )

        retained = retained_share(df, d, mask, product=None)
        table3_rows.append(result_row(sample_name, res, retained))

table3 = pd.DataFrame(table3_rows)
table3


,Sample,d,Inner rho0,Outer rho1,c,Retained windows,p(rho1=rho0)
0,Full sample,12,-0.256***,-0.462***,0.090,100.0%,0.000
1,Full sample,24,-0.455***,-0.725***,0.080,100.0%,0.000
2,Full sample,36,-0.449***,-0.865***,0.046,100.0%,0.000
3,Full sample,48,-0.421**,-0.970***,0.032,100.0%,0.000
4,Local price fixed,12,0.052,-0.094***,0.044,82.2%,0.000
5,Local price fixed,24,0.090,-0.198***,0.042,66.7%,0.000
6,Local price fixed,36,0.087,-0.377***,0.034,52.6%,0.000
7,Local price fixed,48,0.227,-0.660***,0.021,40.7%,0.000
8,US price fixed,12,-0.224***,-0.400***,0.091,93.0%,0.000
9,US price fixed,24,-0.376***,-0.633***,0.084,86.1%,0.000


# Table 4 — Pooled panel sensitivity checks

In [12]:
table4_rows = []

sensitivity_defs = [
    ("Pre-COVID", df, "", lambda base, d: base["date"] < COVID_SPLIT_DATE),
    ("Post-COVID", df, "", lambda base, d: base["date"] >= COVID_SPLIT_DATE),
    (
        "Exclude BR/MX/RU/TR",
        df,
        "",
        lambda base, d: ~base["country"].isin(HIGH_RESET_COUNTRIES),
    ),
    (
        "Winsorized 1%",
        df_w,
        "_w",
        lambda base, d: pd.Series(True, index=base.index),
    ),
]

for sample_name, dat, suffix, mask_fun in sensitivity_defs:
    for d in D_WINDOWS:
        mask = mask_fun(dat, d)

        res = threshold_regression(
            dat,
            d,
            sample_mask=mask,
            product=None,
            pooled=True,
            q_suffix=suffix,
        )

        row = result_row(sample_name, res, retained=None)
        row.pop("Retained windows", None)
        table4_rows.append(row)

table4 = pd.DataFrame(table4_rows)
table4


,Sample,d,Inner rho0,Outer rho1,c,p(rho1=rho0)
0,Pre-COVID,12,-0.246***,-0.541***,0.103,0.000
1,Pre-COVID,24,-0.404***,-0.798***,0.080,0.000
2,Pre-COVID,36,-0.494***,-0.947***,0.061,0.000
3,Pre-COVID,48,-1.248***,-0.951***,0.110,0.001
4,Post-COVID,12,-0.195**,-0.285***,0.058,0.092
5,Post-COVID,24,-0.293,-0.494***,0.043,0.018
6,Post-COVID,36,-0.199,-0.661***,0.034,0.000
7,Post-COVID,48,0.010,-0.825***,0.025,0.000
8,Exclude BR/MX/RU/TR,12,-0.191***,-0.430***,0.077,0.000
9,Exclude BR/MX/RU/TR,24,-0.384***,-0.707***,0.078,0.000


## Export all tables to Excel and LaTeX

In [13]:
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    table1_pretty.to_excel(writer, sheet_name="Table 1 Coverage", index=False)
    table2.to_excel(writer, sheet_name="Table 2 No Reset", index=False)
    table3.to_excel(writer, sheet_name="Table 3 Identification", index=False)
    table4.to_excel(writer, sheet_name="Table 4 Sensitivity", index=False)

with open(OUTPUT_LATEX, "w", encoding="utf-8") as f:
    f.write("% Auto-generated robustness tables\\n\\n")

    f.write("% Table 1\\n")
    f.write(table1_pretty.to_latex(index=False, escape=False))
    f.write("\\n\\n")

    f.write("% Table 2\\n")
    f.write(table2.to_latex(index=False, escape=False))
    f.write("\\n\\n")

    f.write("% Table 3\\n")
    f.write(table3.to_latex(index=False, escape=False))
    f.write("\\n\\n")

    f.write("% Table 4\\n")
    f.write(table4.to_latex(index=False, escape=False))
    f.write("\\n\\n")

print("Wrote:", OUTPUT_XLSX)
print("Wrote:", OUTPUT_LATEX)


Wrote: ipad_robustness_tables_full_replication.xlsx
Wrote: ipad_robustness_tables_full_replication.tex


## Optional diagnostic checks

These are useful if any table unexpectedly has blank rows.


In [14]:
# Diagnostic: number of valid windows by horizon
diagnostics = []
for d in D_WINDOWS:
    diagnostics.append({
        "d": d,
        "valid_windows": int(df[f"valid_window_{d}"].sum()),
        "both_prices_fixed": int((df[f"valid_window_{d}"] & df[f"both_prices_fixed_{d}"]).sum()),
        "same_model_window": int((df[f"valid_window_{d}"] & df[f"same_model_window_{d}"]).sum()),
    })

pd.DataFrame(diagnostics)


,d,valid_windows,both_prices_fixed,same_model_window
0,12,40664,33338,38214
1,24,39032,25879,34542
2,36,37400,19478,30870
3,48,35768,14338,27198
